# Bike accessibility from Brookline Village

How far can you get from home in 5 / 10 / 15 / 20 minutes on a regular bike vs
an e-bike?

**Simplest definition of accessibility:** a straight-line ("as the crow flies")
circle whose radius is `average riding speed × time`. No street network, hills,
or stops — just a first-order sense of reach. See the last cell for ways to
make it more realistic.

In [ ]:
from pathlib import Path

import folium
import yaml

from shared.plots.style import PALETTE

## Params

In [ ]:
HOME = (42.3328, -71.1167)  # approx. Brookline Village (MBTA Green Line D stop)
HOME_LABEL = "Brookline Village"

# Average door-to-door urban riding speeds (km/h), including some slowing for
# intersections. Rough rules of thumb, not measured:
#   - conventional bike ~ 12 km/h (~7.5 mph)
#   - pedal-assist e-bike ~ 18 km/h (~11 mph); US Class 1/3 assist caps at 20/28 mph
SPEEDS_KMH = {"Bike": 12, "E-bike": 18}
MINUTES = [5, 10, 15, 20]
DEFAULT_MINUTES = 10  # which time band is shown when the map opens

COLORS = {"Bike": PALETTE[0], "E-bike": PALETTE[1]}

## Radii

In [ ]:
def radius_m(speed_kmh, minutes):
    return speed_kmh * 1000 * minutes / 60


for minutes in MINUTES:
    row = ", ".join(
        f"{mode}: {radius_m(speed, minutes) / 1000:.1f} km ({radius_m(speed, minutes) / 1609.34:.1f} mi)"
        for mode, speed in SPEEDS_KMH.items()
    )
    print(f"{minutes:>2} min -> {row}")

## Map

Each time band is its own layer, wired up as radio buttons in the layer control
(top right) so only one band — i.e. two circles, bike + e-bike — shows at a
time. The e-bike circle is drawn first so the smaller bike circle sits on top.

In [ ]:
# bike_accessibility/ -> adhoc/ -> analyses/ -> repo root
CONFIG_SECRETS_PATH = Path.cwd().resolve().parents[2] / "config_secrets.yml"


def load_carto_api_key():
    if not CONFIG_SECRETS_PATH.exists():
        print("Warning: config_secrets.yml not found; CARTO map tiles may be rate-limited.")
        return None
    key = (yaml.safe_load(CONFIG_SECRETS_PATH.read_text()) or {}).get("api_keys", {}).get("carto")
    if not key:
        print("Warning: no CARTO api key set in config_secrets.yml; map tiles may be rate-limited.")
    return key


tiles = "https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png"
carto_api_key = load_carto_api_key()
if carto_api_key:
    tiles += f"?key={carto_api_key}"

access_map = folium.Map(location=HOME, zoom_start=12, tiles=None)
# control=False keeps the basemap out of the radio list, so the only choices are the time bands
folium.TileLayer(tiles=tiles, attr="CARTO", name="CARTO Positron", control=False).add_to(access_map)

for minutes in MINUTES:
    # overlay=False -> rendered as mutually exclusive "base layer" radio buttons
    band = folium.FeatureGroup(name=f"{minutes} min", overlay=False, show=(minutes == DEFAULT_MINUTES))
    for mode in sorted(SPEEDS_KMH, key=SPEEDS_KMH.get, reverse=True):
        r = radius_m(SPEEDS_KMH[mode], minutes)
        folium.Circle(
            location=HOME,
            radius=r,
            color=COLORS[mode],
            weight=2,
            fill=True,
            fill_color=COLORS[mode],
            fill_opacity=0.15,
            tooltip=f"{mode}, {minutes} min: {r / 1000:.1f} km ({r / 1609.34:.1f} mi) @ {SPEEDS_KMH[mode]} km/h",
        ).add_to(band)
    band.add_to(access_map)

folium.Marker(HOME, tooltip=HOME_LABEL, icon=folium.Icon(color="gray", icon="home")).add_to(access_map)
folium.LayerControl(collapsed=False).add_to(access_map)

legend_items = "".join(
    f'<div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;'
    f'background:{color};margin-right:6px;"></span>{mode} ({SPEEDS_KMH[mode]} km/h)</div>'
    for mode, color in COLORS.items()
)
access_map.get_root().html.add_child(
    folium.Element(
        f'<div style="position: fixed; bottom: 20px; left: 20px; z-index: 9999; background: white; '
        f'padding: 10px 12px; border: 1px solid #ccc; border-radius: 4px; font-size: 12px; '
        f'font-family: sans-serif;">{legend_items}</div>'
    )
)

out_dir = Path("outputs")
out_dir.mkdir(parents=True, exist_ok=True)
access_map.save(out_dir / "bike_accessibility_map.html")
access_map

## Making it more accurate

Roughly in order of effort vs payoff:

1. **Circuity factor (quick win).** Streets don't go in straight lines. Divide
   each radius by a detour factor (~1.2–1.4 for Boston-area grids) to turn
   "distance ridden" into "straight-line reach".
2. **Network isochrones with OSMnx (already a repo dependency).** Pull the
   OSM bike network (`ox.graph_from_point(HOME, network_type="bike")`), set
   edge travel time = length / speed, and take everything reachable within
   N minutes from the nearest node (`nx.ego_graph(..., distance="time")`).
   Wrap the reached nodes/edges in a concave hull or buffer. This captures
   the river, the Emerald Necklace, the Pike/rail cuts, and missing bridges —
   the big errors in a circle.
3. **Use a routing engine instead.** OpenRouteService (`cycling-regular`,
   `cycling-electric` profiles), Valhalla, or GraphHopper return isochrone
   polygons directly and already model elevation, turn costs, and road
   preferences.
4. **Elevation.** Add node elevations (`ox.elevation` with a DEM) and slow
   conventional bikes on climbs (e.g. Fisher Hill, Corey Hill, Mission Hill)
   — the main place bike and e-bike reach actually diverge most.
5. **Stress-aware routing.** Weight edges by Level of Traffic Stress (bike
   lanes / separated paths vs. arterials like Route 9 or Huntington Ave) so
   "accessible" means "comfortably rideable", not just "legal to ride".
6. **Intersection delay.** Add a per-signal penalty (~15–30 s) on edges with
   traffic signals, which matters a lot at 5–10 minutes.
7. **Calibrate speeds with real trips.** Bluebikes trip data (see
   `analyses/runbooks/bluebikes/`) includes start/end times and stations,
   and the system has both classic and e-bikes — fit actual speeds per bike
   type, time of day, and distance instead of guessing 12 vs 18 km/h. Your
   own Strava/GPS rides would be even better.
8. **Destinations, not just area.** Once you have isochrones, count what's
   inside them (jobs, groceries, parks, T stops) — that's accessibility in
   the planning sense rather than just reach.